# Task 2b: Specialist Restoration Autoencoders (Simple Baseline)

This notebook trains the **three specialist restoration autoencoders** independently:
1. **Salt-and-Pepper Specialist**
2. **Gaussian Blur Specialist**
3. **Occlusion Specialist**

Each specialist is trained strictly on its designated corruption domain (Plan 6 §2b.1).

In [ ]:
# 1. Config
BATCH_SIZE = 32
LR = 1e-3
LATENT_DIM = 128
BASE_CHANNELS = 32
ALPHA = 0.8
BETA = 0.001
EPOCHS = 15
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_vae import ConvVAE
from src.train_loop import train_one_epoch_vae, validate_vae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

In [ ]:
# 3. Specialist Training Function
def train_specialist(corruption_type: str, epochs: int = EPOCHS):
    print(f"\n==============================================")
    print(f"   Training Specialist: {corruption_type.upper()}")
    print(f"==============================================")
    
    train_ds = PetDataset(mode="train", corruption_mode=corruption_type, seed=SEED)
    val_ds = PetDataset(mode="val", corruption_mode=corruption_type, seed=SEED)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    model = ConvVAE(
        in_channels=3,
        base_channels=BASE_CHANNELS,
        latent_dim=LATENT_DIM,
        dropout=0.1,
        use_skip=False
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
    best_loss = float("inf")
    
    for epoch in range(1, epochs + 1):
        tr = train_one_epoch_vae(model, train_loader, optimizer, device, ALPHA, BETA, scaler)
        vl = validate_vae(model, val_loader, device, ALPHA, BETA)
        
        is_best = vl["val_loss"] < best_loss
        if is_best:
            best_loss = vl["val_loss"]
            
        ckpt = create_checkpoint_dict(model, optimizer, epoch, vl["val_loss"], random_seed=SEED)
        save_checkpoint(
            checkpoint_dir=checkpoint_dir,
            prefix=f"task2b_specialist_{corruption_type}",
            ckpt_dict=ckpt,
            is_best=is_best,
            task_name=f"task2b_{corruption_type}",
            phase_name="simple",
            device_name=device_report()["device_name"]
        )
        
        print(f"[{corruption_type.upper()} Epoch {epoch:02d}] Val Loss: {vl['val_loss']:.4f}, SSIM: {vl['val_ssim']:.4f} {'*' if is_best else ''}")
        
    return model

In [ ]:
# 4. Train All Three Specialists
salt_model = train_specialist("salt", epochs=EPOCHS)
blur_model = train_specialist("blur", epochs=EPOCHS)
occl_model = train_specialist("occlusion", epochs=EPOCHS)
print("\nAll 3 specialists trained and checkpoints saved successfully!")

In [ ]:
# 5. Visual Spot-Check
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
corruptions = ["salt", "blur", "occlusion"]
models = [salt_model, blur_model, occl_model]

for idx, (ctype, mdl) in enumerate(zip(corruptions, models)):
    mdl.eval()
    ds = PetDataset(mode="val", corruption_mode=ctype, seed=SEED)
    corrupted, clean = ds[0]
    with torch.no_grad():
        recon, _, _ = mdl(corrupted.unsqueeze(0).to(device))
        
    axes[idx, 0].imshow(corrupted.permute(1, 2, 0).clip(0, 1))
    axes[idx, 0].set_title(f"{ctype.capitalize()} Input")
    axes[idx, 0].axis("off")
    
    axes[idx, 1].imshow(recon.squeeze(0).cpu().permute(1, 2, 0).clip(0, 1))
    axes[idx, 1].set_title("Specialist Recon")
    axes[idx, 1].axis("off")
    
    axes[idx, 2].imshow(clean.permute(1, 2, 0).clip(0, 1))
    axes[idx, 2].set_title("Clean Target")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()